# Experiment 4 — Masking Noise Predictions at Periodic Timesteps

Assess how zeroing the model's **predicted noise** ($\hat\varepsilon$) at selected reverse steps affects the final image, with separate control over the **conditional** and **unconditional** CFG branches.

**Protocol** (repeated for every KID-tier class)
1. Fix the initial latent and per-step sampling noise so comparisons differ only by which $\hat\varepsilon$ branches are masked.
2. At reverse steps every `every_n_steps` (default 50, 100, 150, 200), zero the predicted noise on:
   - **conditional** branch only,
   - **unconditional** branch only,
   - **both** branches, or
   - **neither** (baseline).
3. Decode finals and report pairwise distance vs the unmasked baseline (CLIP / DINOv2, LPIPS, latent MSE).
4. Aggregate across **KID difficulty tiers** (15 hardest / 15 medium / 15 easiest); detailed grids for one spotlight class per tier.


# 1. Setup


In [ ]:
%cd ../


In [ ]:
#!git clone https://github.com/facebookresearch/DiT.git
import DiT, os
os.chdir("DiT")

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from torchvision.utils import make_grid, save_image

from diffusion import create_diffusion
from diffusers.models import AutoencoderKL
from download import find_model
from models import DiT_XL_2

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={device}")
if device == "cpu":
    print("GPU not found. Using CPU instead.")


# 2. Load DiT-XL/2


In [ ]:
image_size = 256  #@param [256, 512]
vae_model = "stabilityai/sd-vae-ft-ema"  #@param ["stabilityai/sd-vae-ft-mse", "stabilityai/sd-vae-ft-ema"]
latent_size = int(image_size) // 8

model = DiT_XL_2(input_size=latent_size).to(device)
state_dict = find_model(f"DiT-XL-2-{image_size}x{image_size}.pt")
model.load_state_dict(state_dict)
model.eval()
vae = AutoencoderKL.from_pretrained(vae_model).to(device)
print("Model + VAE ready.")


# 3. Config — KID difficulty tiers

Classes are taken from the same ranking used to build `results/ensemble_K50_fid/kid_difficulty_tiers.png`: 15 hardest / 15 medium / 15 easiest by per-class KID.


In [ ]:
# ---- Sampling ----
cfg_scale = 4.0  #@param {type:"slider", min:1, max:10, step:0.1}
num_sampling_steps = 250  #@param {type:"slider", min:10, max:1000, step:1}
traj_seed = 0  #@param {type:"integer"}
denoise_seed = 12345  #@param {type:"integer"}  # shared per-step sampling noise across all mask modes

# Mask predicted noise (eps) at these reverse-step indices (every N steps).
every_n_steps = 50  #@param {type:"integer"}

# Which mask modes to run. Options: none, conditional, unconditional, both
mask_modes_to_run = ("none", "conditional", "unconditional", "both")  #@param {type:"raw"}

# Checkpoints shown in spotlight image grids
spotlight_show_steps = (100, 150)  #@param {type:"raw"}

# ---- KID difficulty tiers (same split as kid_difficulty_tiers.png) ----
KID_JSON = Path("../results/ensemble_K50_fid/kid_per_class.json")
IMAGENET_IDX = Path("../results/ensemble_K50_fid/imagenet_class_index.json")

with open(KID_JSON) as f:
    kid_per_class = {int(k): v for k, v in json.load(f).items()}
with open(IMAGENET_IDX) as f:
    _idx = json.load(f)
class_names = {i: _idx[str(i)][1] for i in range(len(_idx))}

kid_ranked = sorted(kid_per_class.keys(), key=lambda c: kid_per_class[c]["kid"], reverse=True)
n_ranked = len(kid_ranked)
TIER_SIZE = min(15, n_ranked // 3 if n_ranked >= 3 else n_ranked)

hardest_classes = kid_ranked[:TIER_SIZE]
easiest_classes = kid_ranked[-TIER_SIZE:] if TIER_SIZE > 0 else []
mid_start = max(0, n_ranked // 2 - TIER_SIZE // 2)
medium_classes = kid_ranked[mid_start: mid_start + TIER_SIZE]

tiers = {
    "hardest": hardest_classes,
    "medium": medium_classes,
    "easiest": easiest_classes,
}
tier_colors = {
    "hardest": "crimson",
    "medium": "goldenrod",
    "easiest": "seagreen",
}

spotlight_classes = {name: cls_list[len(cls_list) // 2] for name, cls_list in tiers.items()}
class_labels = hardest_classes + medium_classes + easiest_classes
class_to_tier = {}
for name, cls_list in tiers.items():
    for c in cls_list:
        class_to_tier[c] = name

RESULTS_DIR = Path("../results/exp4_noise_mask_prediction")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

diffusion = create_diffusion(str(num_sampling_steps))
shape = (1, 4, latent_size, latent_size)
T = diffusion.num_timesteps
assert T == num_sampling_steps, (T, num_sampling_steps)

mask_steps = tuple(range(every_n_steps, T, every_n_steps))
save_steps = set(mask_steps) | {0, T}
spotlight_show_steps = tuple(int(s) for s in spotlight_show_steps)
assert all(s in mask_steps for s in spotlight_show_steps), (
    f"spotlight_show_steps={spotlight_show_steps} must be subset of mask_steps={mask_steps}"
)

MASK_BRANCH_MAP = {
    "none": set(),
    "conditional": {"cond"},
    "unconditional": {"uncond"},
    "both": {"cond", "uncond"},
}
for m in mask_modes_to_run:
    assert m in MASK_BRANCH_MAP, f"unknown mask mode: {m}"

print(f"steps={T}, cfg={cfg_scale}, mask_steps={mask_steps}")
print(f"modes={mask_modes_to_run}")
print(f"traj_seed={traj_seed}, denoise_seed={denoise_seed}")
print(f"\nTIER_SIZE={TIER_SIZE} | total classes to run = {len(class_labels)}")
for name, cls_list in tiers.items():
    spot = spotlight_classes[name]
    print(f"\n=== {name} ({len(cls_list)}) | spotlight={spot} ({class_names.get(spot, '?')}) ===")
    for c in cls_list:
        print(f"  class {c:>4}  KID={kid_per_class[c]['kid']:.4f}  {class_names.get(c, '?')}")
print(f"\nresults -> {RESULTS_DIR.resolve()}")


# 4. Diffusion helpers

At masked reverse steps we zero the model's predicted $\hat\varepsilon$ (first 3 channels) on the selected CFG branch(es) **before** classifier-free guidance is applied. Per-step **sampling** noise $\varepsilon$ is unchanged and shared across modes via `denoise_seed`.

Indexing convention:
- Reverse step $s=0$: initial noise.
- After $s$ reverse steps, latent `traj[s]` uses spaced model timestep $i = T-1-s`.


In [ ]:
def spaced_index_at_step(s):
    return T - 1 - s


def decode_latents(latents):
    return vae.decode(latents / 0.18215).sample


def save_uint8_image(tensor_chw, path):
    x = torch.clamp(127.5 * tensor_chw + 128.0, 0, 255)
    x = x.permute(1, 2, 0).to("cpu", dtype=torch.uint8).numpy()
    Image.fromarray(x).save(path)


def cfg_kwargs(class_label):
    y = torch.tensor([class_label, 1000], device=device)
    return dict(y=y, cfg_scale=cfg_scale)


_sampling_state = {"reverse_step": 0, "mask_mode": "none", "mask_steps": mask_steps}


def branches_to_mask(mask_mode, reverse_step):
    if reverse_step not in _sampling_state["mask_steps"]:
        return set()
    return MASK_BRANCH_MAP[mask_mode]


def forward_with_masked_cfg(x, t, y, cfg_scale):
    """CFG forward that can zero cond/uncond eps predictions at selected reverse steps."""
    half = x[: len(x) // 2]
    combined = torch.cat([half, half], dim=0)
    model_out = model.forward(combined, t, y)
    eps, rest = model_out[:, :3], model_out[:, 3:]
    cond_eps, uncond_eps = torch.split(eps, len(eps) // 2, dim=0)

    masked = branches_to_mask(_sampling_state["mask_mode"], _sampling_state["reverse_step"])
    if "cond" in masked:
        cond_eps = torch.zeros_like(cond_eps)
    if "uncond" in masked:
        uncond_eps = torch.zeros_like(uncond_eps)

    half_eps = uncond_eps + cfg_scale * (cond_eps - uncond_eps)
    eps = torch.cat([half_eps, half_eps], dim=0)
    return torch.cat([eps, rest], dim=1)


def p_sample_with_noise(diffusion, model_fn, x, t, step_noise, model_kwargs):
    out = diffusion.p_mean_variance(
        model_fn, x, t, clip_denoised=False, model_kwargs=model_kwargs
    )
    nonzero_mask = (t != 0).float().view(-1, *([1] * (len(x.shape) - 1)))
    sample = out["mean"] + nonzero_mask * torch.exp(0.5 * out["log_variance"]) * step_noise
    return {"sample": sample, "pred_xstart": out["pred_xstart"]}


def precompute_step_noise(noise_seed):
    g = torch.Generator(device=device)
    g.manual_seed(int(noise_seed))
    return [
        torch.randn(2, 4, latent_size, latent_size, generator=g, device=device)
        for _ in range(T)
    ]


def generate_with_mask(class_label, seed, step_noises, mask_mode, save_steps=None):
    """Full reverse path with optional eps-branch masking.

    Returns dict reverse_step -> latent (1,C,H,W) on CPU.
    """
    save_set = set(save_steps or []) | {0, T}
    torch.manual_seed(seed)
    z = torch.randn(*shape, device=device)
    img = torch.cat([z, z], dim=0)
    model_kwargs = cfg_kwargs(class_label)

    _sampling_state["mask_mode"] = mask_mode
    traj = {}
    if 0 in save_set:
        traj[0] = img[:1].detach().cpu()

    indices = list(range(T))[::-1]
    for step_i, i in enumerate(tqdm(indices, desc=f"c{class_label} mask={mask_mode}", leave=False)):
        _sampling_state["reverse_step"] = step_i
        t = torch.tensor([i, i], device=device)
        out = p_sample_with_noise(
            diffusion,
            forward_with_masked_cfg,
            img,
            t,
            step_noises[step_i],
            model_kwargs,
        )
        img = out["sample"]
        s_after = step_i + 1
        if s_after in save_set:
            traj[s_after] = img[:1].detach().cpu()

    return traj


print("Diffusion helpers ready.")


# 5. Metric helpers (pairwise vs baseline)


In [ ]:
import open_clip
import lpips
from transformers import AutoImageProcessor, AutoModel

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai"
)
clip_model = clip_model.to(device).eval()

dino_name = "facebook/dinov2-small"
dino_processor = AutoImageProcessor.from_pretrained(dino_name)
dino_model = AutoModel.from_pretrained(dino_name).to(device).eval()

lpips_fn = lpips.LPIPS(net="alex").to(device).eval()


def _images_to_pil_list(imgs_bchw):
    x = torch.clamp(imgs_bchw * 0.5 + 0.5, 0, 1)
    out = []
    for i in range(x.shape[0]):
        arr = (x[i].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        out.append(Image.fromarray(arr))
    return out


@torch.no_grad()
def embed_clip(imgs_bchw):
    pils = _images_to_pil_list(imgs_bchw)
    tensors = torch.stack([clip_preprocess(p) for p in pils]).to(device)
    feats = clip_model.encode_image(tensors)
    return F.normalize(feats.float(), dim=-1).cpu()


@torch.no_grad()
def embed_dino(imgs_bchw):
    pils = _images_to_pil_list(imgs_bchw)
    inputs = dino_processor(images=pils, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    out = dino_model(**inputs)
    feats = out.last_hidden_state[:, 0]
    return F.normalize(feats.float(), dim=-1).cpu()


@torch.no_grad()
def pairwise_metrics(latent_a, latent_b):
    imgs = decode_latents(torch.cat([latent_a, latent_b], dim=0).to(device)).cpu()
    img_a, img_b = imgs[0:1], imgs[1:2]
    clip_a, clip_b = embed_clip(img_a), embed_clip(img_b)
    dino_a, dino_b = embed_dino(img_a), embed_dino(img_b)
    lp = float(lpips_fn(img_a.to(device), img_b.to(device)).item())
    mse = float(F.mse_loss(latent_a, latent_b).item())
    return {
        "clip_cos_dist": float(1.0 - (clip_a * clip_b).sum().item()),
        "dino_cos_dist": float(1.0 - (dino_a * dino_b).sum().item()),
        "lpips": lp,
        "latent_mse": mse,
    }, img_a[0], img_b[0]


print("Metric helpers ready.")


# 6. Per-class runner

`run_class_experiment(class_label)` runs all mask modes for one class, caches trajectories / metrics under `results/exp4_noise_mask_prediction/class_XXXX/`, and skips work when `metrics_noise_mask.json` already exists.


In [ ]:
def run_class_experiment(class_label, save_visuals=False):
    """Run all mask modes for one class. Returns metrics dict.

    If metrics_noise_mask.json exists, load and return it (skip generation).
    When save_visuals=True (spotlight classes), also write final/grid PNGs.
    """
    samples_dir = RESULTS_DIR / f"class_{class_label:04d}"
    samples_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = samples_dir / "metrics_noise_mask.json"

    if metrics_path.exists() and not save_visuals:
        with open(metrics_path) as f:
            metrics = json.load(f)
        print(f"  [{class_label}] SKIP (cached metrics)")
        return metrics

    step_noises = precompute_step_noise(denoise_seed)
    all_trajs = {}

    for mask_mode in mask_modes_to_run:
        out_path = samples_dir / f"traj_{mask_mode}.pt"
        if out_path.exists():
            all_trajs[mask_mode] = torch.load(out_path, map_location="cpu", weights_only=False)
            print(f"  [{class_label}] loaded traj_{mask_mode}.pt")
        else:
            traj = generate_with_mask(
                class_label,
                traj_seed,
                step_noises,
                mask_mode,
                save_steps=save_steps,
            )
            torch.save(traj, out_path)
            all_trajs[mask_mode] = traj
            print(f"  [{class_label}] saved traj_{mask_mode}.pt")

    baseline_mode = "none" if "none" in all_trajs else mask_modes_to_run[0]
    baseline_final = all_trajs[baseline_mode][T]

    if metrics_path.exists():
        with open(metrics_path) as f:
            metrics = json.load(f)
    else:
        metrics = {
            "config": {
                "class_label": class_label,
                "tier": class_to_tier.get(class_label),
                "cfg_scale": cfg_scale,
                "num_sampling_steps": T,
                "mask_steps": list(mask_steps),
                "mask_modes": list(mask_modes_to_run),
                "traj_seed": traj_seed,
                "denoise_seed": denoise_seed,
                "baseline_mode": baseline_mode,
            },
            "modes": {},
            "checkpoints": {},
        }

        for mask_mode, traj in all_trajs.items():
            if mask_mode == baseline_mode:
                continue
            m, _, _ = pairwise_metrics(traj[T], baseline_final)
            metrics["modes"][mask_mode] = m
            print(f"  [{class_label}] final vs {baseline_mode} | {mask_mode}: "
                  f"clip={m['clip_cos_dist']:.4f} lpips={m['lpips']:.4f}")

            metrics["checkpoints"][mask_mode] = {}
            for s in mask_steps:
                if s in traj and s in all_trajs[baseline_mode]:
                    cm, _, _ = pairwise_metrics(traj[s], all_trajs[baseline_mode][s])
                    metrics["checkpoints"][mask_mode][str(s)] = cm

        with open(metrics_path, "w") as f:
            json.dump(metrics, f, indent=2)
        print(f"  [{class_label}] wrote {metrics_path.name}")

    if save_visuals:
        all_images = {}
        for mask_mode, traj in all_trajs.items():
            img = decode_latents(traj[T].to(device)).cpu()[0]
            all_images[mask_mode] = img
            save_uint8_image(img, samples_dir / f"final_{mask_mode}.png")

        cols = list(all_images.keys())
        grid = make_grid(
            torch.stack([all_images[m] for m in cols]),
            nrow=len(cols),
            normalize=True,
            value_range=(-1, 1),
        )
        save_image(grid, samples_dir / "grid_final_all_modes.png")

        for s in spotlight_show_steps:
            rows, labels = [], []
            rows.append(decode_latents(all_trajs[baseline_mode][s].to(device)).cpu()[0])
            labels.append(baseline_mode)
            for mask_mode in mask_modes_to_run:
                if mask_mode == baseline_mode:
                    continue
                if s in all_trajs[mask_mode]:
                    rows.append(decode_latents(all_trajs[mask_mode][s].to(device)).cpu()[0])
                    labels.append(mask_mode)
            if len(rows) >= 2:
                g = make_grid(torch.stack(rows), nrow=len(rows), normalize=True, value_range=(-1, 1))
                save_image(g, samples_dir / f"grid_checkpoint_s{s:03d}.png")
        print(f"  [{class_label}] saved spotlight visuals")

    return metrics


print("run_class_experiment() ready.")


# 7. Run all KID-tier classes

Loops over hardest / medium / easiest (45 classes). Finished classes are skipped via cached `metrics_noise_mask.json`. Spotlight classes also get PNG grids.


In [ ]:
done = [
    c for c in class_labels
    if (RESULTS_DIR / f"class_{c:04d}" / "metrics_noise_mask.json").exists()
]
pending = [c for c in class_labels if c not in done]
print(f"Progress: {len(done)}/{len(class_labels)} classes done, {len(pending)} remaining")
if device == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"GPU mem free={free/1e9:.2f}G / total={total/1e9:.2f}G")

all_metrics = {}
for tier_name, cls_list in tiers.items():
    for class_label in tqdm(cls_list, desc=f"tier={tier_name}"):
        print(
            f"\n>>> class {class_label} ({class_names.get(class_label, '?')}) "
            f"| tier={tier_name} | KID={kid_per_class[class_label]['kid']:.4f}"
        )
        save_vis = class_label in spotlight_classes.values()
        all_metrics[class_label] = run_class_experiment(class_label, save_visuals=save_vis)

# Aggregate summary JSON (reload from disk so skipped classes are included)
for c in class_labels:
    if c not in all_metrics:
        with open(RESULTS_DIR / f"class_{c:04d}" / "metrics_noise_mask.json") as f:
            all_metrics[c] = json.load(f)

summary = {
    "tiers": {k: list(v) for k, v in tiers.items()},
    "spotlight_classes": spotlight_classes,
    "spotlight_show_steps": list(spotlight_show_steps),
    "mask_steps": list(mask_steps),
    "mask_modes": list(mask_modes_to_run),
    "cfg_scale": cfg_scale,
    "traj_seed": traj_seed,
    "denoise_seed": denoise_seed,
    "per_class": {
        str(c): {
            "tier": class_to_tier[c],
            "kid": kid_per_class[c]["kid"],
            "name": class_names.get(c, "?"),
            "modes": all_metrics[c].get("modes", {}),
            "checkpoints": all_metrics[c].get("checkpoints", {}),
        }
        for c in class_labels
    },
}
summary_path = RESULTS_DIR / "summary_all_tiers.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nWrote {summary_path} ({len(all_metrics)} classes)")


# 8. Tier-aggregated plots

Mean ± std of final-image distance vs baseline across classes within each KID tier, by mask mode. Colors match `kid_difficulty_tiers.png` (crimson / goldenrod / seagreen).


In [ ]:
METRIC_KEYS = ["clip_cos_dist", "dino_cos_dist", "lpips", "latent_mse"]
METRIC_TITLES = ["CLIP cos dist", "DINO cos dist", "LPIPS", "Latent MSE"]
COMPARE_MODES = [m for m in mask_modes_to_run if m != "none"]


def collect_tier_values(tier_name, mode, key, where="modes", step=None):
    vals = []
    for c in tiers[tier_name]:
        m = all_metrics[c]
        if where == "modes":
            if mode in m.get("modes", {}) and key in m["modes"][mode]:
                vals.append(m["modes"][mode][key])
        else:
            ck = m.get("checkpoints", {}).get(mode, {}).get(str(step), {})
            if key in ck:
                vals.append(ck[key])
    return np.asarray(vals, dtype=np.float64)


# ---- Final distance vs baseline: bar per mode, grouped by tier ----
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
x = np.arange(len(COMPARE_MODES))
width = 0.25
for ax, key, title in zip(axes, METRIC_KEYS, METRIC_TITLES):
    for i, (tier_name, color) in enumerate(tier_colors.items()):
        means, stds = [], []
        for mode in COMPARE_MODES:
            vals = collect_tier_values(tier_name, mode, key, where="modes")
            means.append(vals.mean() if len(vals) else np.nan)
            stds.append(vals.std(ddof=0) if len(vals) else np.nan)
        ax.bar(
            x + (i - 1) * width,
            means,
            width,
            yerr=stds,
            label=tier_name,
            color=color,
            capsize=3,
            alpha=0.85,
        )
    ax.set_xticks(x)
    ax.set_xticklabels(COMPARE_MODES, rotation=20)
    ax.set_title(title)
    ax.set_ylabel("distance vs none")
axes[0].legend(fontsize=8)
fig.suptitle("Final-image sensitivity to eps masking by KID tier (mean ± std over classes)")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "tier_bar_final.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- Checkpoint curves: distance vs reverse step, one panel per mode ----
fig, axes = plt.subplots(1, len(COMPARE_MODES), figsize=(4.5 * len(COMPARE_MODES), 4), squeeze=False)
axes = axes[0]
for ax, mode in zip(axes, COMPARE_MODES):
    for tier_name, color in tier_colors.items():
        means, stds = [], []
        for s in mask_steps:
            vals = collect_tier_values(tier_name, mode, "clip_cos_dist", where="checkpoints", step=s)
            means.append(vals.mean() if len(vals) else np.nan)
            stds.append(vals.std(ddof=0) if len(vals) else np.nan)
        means, stds = np.asarray(means), np.asarray(stds)
        ax.plot(mask_steps, means, "-o", color=color, label=tier_name, markersize=4)
        ax.fill_between(mask_steps, means - stds, means + stds, color=color, alpha=0.15)
    ax.set_title(f"mask={mode}")
    ax.set_xlabel("reverse step s")
    ax.set_ylabel("CLIP cos dist vs none")
    ax.legend(fontsize=8)
fig.suptitle("Checkpoint latent distance vs none (CLIP) by mask mode and KID tier")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "tier_curves_checkpoint_clip.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- Boxplots of final LPIPS by tier × mode ----
fig, axes = plt.subplots(1, len(COMPARE_MODES), figsize=(4.2 * len(COMPARE_MODES), 4), squeeze=False)
axes = axes[0]
for ax, mode in zip(axes, COMPARE_MODES):
    data, labels, colors = [], [], []
    for tier_name, color in tier_colors.items():
        vals = collect_tier_values(tier_name, mode, "lpips", where="modes")
        if len(vals):
            data.append(vals)
            labels.append(tier_name)
            colors.append(color)
    bp = ax.boxplot(data, labels=labels, patch_artist=True)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(f"mask={mode}")
    ax.set_ylabel("LPIPS vs none")
    ax.tick_params(axis="x", rotation=15)
fig.suptitle("Final LPIPS distance vs none — distribution over classes per tier")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "tier_boxplots_lpips.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved tier_bar_final.png, tier_curves_checkpoint_clip.png, tier_boxplots_lpips.png")


# 9. Spotlight visuals

Show final grids for the three spotlight (median-KID) classes — one per tier. Metrics for all classes are already in `metrics_noise_mask.json`; this only displays / regenerates PNGs.


In [ ]:
for tier_name, cl in spotlight_classes.items():
    # ensure visuals exist
    run_class_experiment(cl, save_visuals=True)
    samples_dir = RESULTS_DIR / f"class_{cl:04d}"
    print(
        f"\n=== spotlight {tier_name} | class {cl} ({class_names.get(cl, '?')}) "
        f"| KID={kid_per_class[cl]['kid']:.4f} ==="
    )
    grid_path = samples_dir / "grid_final_all_modes.png"
    if grid_path.exists():
        img = Image.open(grid_path)
        fig, ax = plt.subplots(figsize=(12, 3.2))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"{tier_name} | class {cl} ({class_names.get(cl, '?')}) | finals")
        fig.text(0.5, 0.02, " | ".join(mask_modes_to_run), ha="center", fontsize=10)
        fig.tight_layout()
        plt.show()
    for s in spotlight_show_steps:
        p = samples_dir / f"grid_checkpoint_s{s:03d}.png"
        if p.exists():
            img = Image.open(p)
            fig, ax = plt.subplots(figsize=(12, 3.2))
            ax.imshow(img)
            ax.axis("off")
            ax.set_title(f"{tier_name} | class {cl} | checkpoint s={s}")
            fig.tight_layout()
            plt.show()

print("Done.")


## Notes

- **conditional**: zeros $\hat\varepsilon$ from the class-conditioned forward pass only; CFG still uses the unconditional branch.
- **unconditional**: zeros $\hat\varepsilon$ from the null-class forward pass only.
- **both**: zeros both branches before CFG — equivalent to $\hat\varepsilon_{\text{cfg}} = 0$ at masked steps.
- **none**: standard sampling (baseline).

Re-run section 7 after changing `mask_modes_to_run`, `every_n_steps`, or seeds. Delete cached `traj_*.pt` / `metrics_noise_mask.json` to force regeneration for selected classes.
